# Scaling validation vs theoretical forms

This notebook validates fixed-accuracy scaling trends using the same extrapolation inputs as `fig_fixedacc_validation_replot.ipynb`.

It overlays observed empirical points, extrapolated curves from the fitted predictor, and theoretical scaling templates.

In [ ]:
import importlib.util
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Repo root (walk up to .git so this works from any CWD)
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / '.git').exists():
    REPO_ROOT = REPO_ROOT.parent

MPLSTYLE = REPO_ROOT / "manuscript" / "figures_notebooks" / "single_column.mplstyle"

OUT_DIR = REPO_ROOT / "data" / "curve_fitting" / "plots_fixed_acc_validation_from_B1600"
SCALING_DIR = OUT_DIR / "scaling_validation"
SCALING_DIR.mkdir(parents=True, exist_ok=True)

ROWS_JSON = {
    "hypergraph": REPO_ROOT / "data" / "curve_fitting/plots_unified_paper_full/ci68/hg_bootstrap_threshold_rows.json",
    "eigenshadow": REPO_ROOT / "data" / "curve_fitting/plots_unified_paper_full/ci68/eigenshadow_bootstrap_threshold_rows.json",
    "ml": REPO_ROOT / "data" / "curve_fitting/plots_unified_paper_full/ci68/ml_bootstrap_threshold_rows.json",
}

channels = ["dephasing", "relaxation", "depolarizing"]
amplitudes = [0.01, 0.05, 0.1]
targets = [0.8]
nq_min, nq_max = 5, 50
ci_z = 1.0
nq_grid = np.arange(nq_min, nq_max + 1)


In [43]:
spec = importlib.util.spec_from_file_location(
    "plot_mf_fixed_acc_validation",
    REPO_ROOT / "data" / "curve_fitting" / "plot_mf_fixed_acc_validation.py",
)
pfa = importlib.util.module_from_spec(spec)
sys.modules["plot_mf_fixed_acc_validation"] = pfa
spec.loader.exec_module(pfa)

unified = pfa._load_module("unified_fixed_acc_validation_mod", pfa.UNIFIED_PATH)

rows_by_case, predictors = {}, {}
for method in ["hypergraph", "eigenshadow"]:
    preloaded = unified._load_rows_json(ROWS_JSON[method])
    for ch in channels:
        for amp in amplitudes:
            rows = pfa._filter_rows_case(preloaded, channel=str(ch), amplitude=float(amp))
            rows_by_case[(method, ch, float(amp))] = rows
            predictors[(method, ch, float(amp))] = pfa._build_predictor_for_method(unified, rows, method)

print("Prepared predictors for", len(predictors), "cases")

Prepared predictors for 18 cases


In [44]:
def gamma_bar(channel: str, eps_p: float) -> float:
    eps_p = float(eps_p)
    if channel == "dephasing":
        g_act = 1.0 - 2.0 * eps_p
        g_pass = 1.0
    elif channel == "relaxation":
        g_act = np.sqrt(max(1.0 - eps_p, 1e-12))
        g_pass = 1.0 - eps_p / 2.0
    elif channel == "depolarizing":
        g_act = 1.0 - 4.0 * eps_p / 3.0
        g_pass = 1.0 - 2.0 * eps_p / 3.0
    else:
        raise ValueError(f"Unknown channel: {channel}")
    return 0.5 * (g_act + g_pass)


def theory_hypergraph_relaxation(
    nq: np.ndarray, eps_p: float, lam: float, eps_target: float = 0.2
) -> np.ndarray:
    nq = np.asarray(nq, dtype=float)
    base = 2.0 / max(1.0 - lam * float(eps_p), 1e-12)
    return nq * np.power(base, nq) * np.log(np.maximum(nq / eps_target, 1.0001))


def theory_curve(
    method: str,
    channel: str,
    eps_p: float,
    nq: np.ndarray,
    *,
    lambda_relax: float = 3.0,
) -> np.ndarray:
    nq = np.asarray(nq, dtype=float)
    eps_target = 0.2
    if method == "eigenshadow":
        gbar = gamma_bar(channel, eps_p)
        base = 2.5 / max(gbar * gbar, 1e-12)
        return np.power(base, nq)
    if method == "hypergraph":
        if channel in {"dephasing", "depolarizing"}:
            return nq * np.power(2.0, nq) * np.log(np.maximum(nq / eps_target, 1.0001)) / max(2.0 - eps_p, 1e-12)
        if channel == "relaxation":
            return theory_hypergraph_relaxation(nq, eps_p, float(lambda_relax), eps_target)
        return nq * np.power(2.0, nq) * np.log(np.maximum(nq / eps_target, 1.0001))
    raise ValueError(f"Unsupported method for this notebook: {method}")


def fit_prefactor(y_data: np.ndarray, y_model: np.ndarray) -> float:
    mask = np.isfinite(y_data) & np.isfinite(y_model) & (y_data > 0) & (y_model > 0)
    if np.count_nonzero(mask) < 2:
        return np.nan
    # Fit multiplicative constant c: log(y_data) ~= log(c) + log(y_model)
    return float(np.exp(np.mean(np.log(y_data[mask]) - np.log(y_model[mask]))))


def fit_lambda_hypergraph_relaxation(eps_p: float, nq: np.ndarray, y: np.ndarray) -> tuple[float, float]:
    """Fit (lambda, multiplicative prefactor) for hypergraph relaxation template."""
    from scipy.optimize import minimize_scalar

    eps_p = float(eps_p)
    nq = np.asarray(nq, dtype=float)
    y = np.asarray(y, dtype=float)
    upper = max(1.0 / eps_p * 0.999 - 1e-9, 1e-3)

    def sqerr(lam: float) -> float:
        if lam <= 0:
            return np.inf
        y_m = theory_hypergraph_relaxation(nq, eps_p, lam)
        c = fit_prefactor(y, y_m)
        if not np.isfinite(c) or c <= 0:
            return np.inf
        y_pred = c * y_m
        mask = np.isfinite(y_pred) & (y_pred > 0) & np.isfinite(y) & (y > 0)
        if np.count_nonzero(mask) < 2:
            return np.inf
        return float(np.sum((np.log(y[mask]) - np.log(y_pred[mask])) ** 2))

    res = minimize_scalar(sqerr, bounds=(1e-4, upper), method="bounded")
    lam_hat = float(res.x)
    y_m = theory_hypergraph_relaxation(nq, eps_p, lam_hat)
    c_hat = fit_prefactor(y, y_m)
    return lam_hat, c_hat

In [45]:
plt.style.use(str(MPLSTYLE))

methods = ["hypergraph", "eigenshadow"]

# Match app_hyper notebook default Matplotlib cycle colors (C0, C1, C2)
EPS_COLORS = {0.01: "C0", 0.05: "C1", 0.1: "C2"}


def panel_formula_text(method: str, channel: str) -> str:
    if method == "eigenshadow":
        return r"$n_c \propto \left(\frac{2.5}{\bar{\gamma}^2}\right)^{n_q}$" + "\n" + r"$\bar{\gamma}=\frac{\bar{\gamma}_{act}+\bar{\gamma}_{pass}}{2}$"
    if method == "hypergraph" and channel in {"dephasing", "depolarizing"}:
        return r"$n_c \propto \frac{n_q 2^{n_q}}{2-\epsilon_p}\ln\frac{n_q}{0.2}$"
    if method == "hypergraph" and channel == "relaxation":
        return (
            r"$n_c \propto n_q\left(\frac{2}{1-\hat{c}\epsilon_p}\right)^{n_q}\ln\frac{n_q}{0.2}$"
            + "\n"
            + r"($\hat{c}$ fit per $\epsilon_p$)"
        )
    return r"$n_c$ theory template"


def plot_method_panels(method, ax_row, all_y, hyper_relax_c):
    for j, ch in enumerate(channels):
        ax = ax_row[j]
        for amp in amplitudes:
            col = EPS_COLORS[float(amp)]
            pred = predictors[(method, ch, float(amp))]
            rows = rows_by_case[(method, ch, float(amp))]
            if pred is None or not rows:
                continue

            obs_nq = pfa._obs_nq_by_method(unified, method, rows, nq_min=nq_min)
            if not obs_nq:
                continue
            obs_max = int(max(obs_nq))

            for target in targets:
                x_obs, y_obs = pfa._extract_observed_curve(rows, obs_nq, float(target))
                x_ext, y_ext, y_lo, y_hi, _ = pfa._predict_extrapolated_curve(
                    unified, pred, obs_max, nq_grid, float(target), float(ci_z)
                )

                x_all = np.concatenate([x_obs, x_ext]) if (x_obs.size + x_ext.size) else np.array([])
                y_all = np.concatenate([y_obs, y_ext]) if (y_obs.size + y_ext.size) else np.array([])
                if x_all.size == 0:
                    continue

                if method == "hypergraph" and ch == "relaxation":
                    lam_hat, c_fit = fit_lambda_hypergraph_relaxation(float(amp), x_all, y_all)
                    if hyper_relax_c is not None:
                        hyper_relax_c[float(amp)] = lam_hat
                    y_th = c_fit * theory_curve(method, ch, float(amp), nq_grid, lambda_relax=lam_hat)
                else:
                    y_model = theory_curve(method, ch, float(amp), x_all)
                    c_fit = fit_prefactor(y_all, y_model)
                    y_th = c_fit * theory_curve(method, ch, float(amp), nq_grid)

                if x_obs.size:
                    ax.plot(x_obs, y_obs, "o-", color=col, linewidth=1.6, markersize=4.0)
                    all_y.extend(y_obs.tolist())
                if x_ext.size:
                    rgba_fc = (*plt.matplotlib.colors.to_rgb(col), 0.35)
                    ax.scatter(
                        x_ext,
                        y_ext,
                        marker="^",
                        s=38,
                        facecolors=rgba_fc,
                        edgecolors=col,
                        linewidths=0.9,
                        zorder=5,
                    )
                    valid_ci = np.isfinite(y_lo) & np.isfinite(y_hi) & (y_lo > 0) & (y_hi > 0)
                    if np.any(valid_ci):
                        ax.fill_between(x_ext[valid_ci], y_lo[valid_ci], y_hi[valid_ci], color=col, alpha=0.18, linewidth=0)
                    all_y.extend(y_ext.tolist())

                if np.isfinite(c_fit):
                    ax.plot(nq_grid, y_th, linestyle=":", color=col, alpha=0.9, linewidth=1.3)
                    all_y.extend(y_th[np.isfinite(y_th)].tolist())

        ax.set_yscale("log")
        ax.set_xlim(nq_min - 0.5, nq_max + 0.5)
        ax.grid(True, linestyle="--", alpha=0.35)
        ax.set_xlabel(r"Number of qubits $n_q$")
        if j == 0:
            ax.set_ylabel(r"Sample complexity $n_c$")
        ax.set_title(ch.title())
        ax.text(
            0.98,
            0.03,
            panel_formula_text(method, ch),
            transform=ax.transAxes,
            fontsize=8.5,
            va="bottom",
            ha="right",
            bbox=dict(boxstyle="round,pad=0.22", facecolor="white", alpha=0.72, edgecolor="0.6"),
        )


all_y = []
fig_axes_list = []
hyper_relax_c = {}
n_cols = len(channels)
for method in methods:
    fig, axes = plt.subplots(1, n_cols, figsize=(8.0, 3.65), sharex=True, sharey=True)
    axes = np.atleast_1d(axes)
    c_store = hyper_relax_c if method == "hypergraph" else None
    plot_method_panels(method, axes, all_y, c_store)
    fig_axes_list.append((method, fig, axes))

if hyper_relax_c:
    parts = [rf"$\epsilon_p={ep:g}\rightarrow\hat{{c}}={hyper_relax_c[ep]:.4g}$" for ep in sorted(hyper_relax_c)]
    print("Hypergraph | relaxation | fitted ĉ per ε_p:", " ; ".join(parts))

if all_y:
    y_min = max(min(all_y) * 0.7, 1.0)
    y_max = max(all_y) * 1.5
else:
    y_min, y_max = 1.0, 1e6

handles_sem = [
    plt.Line2D([0], [0], color="black", linestyle="-", marker="o", label="observed"),
    plt.Line2D(
        [0],
        [0],
        color="0.35",
        linestyle="None",
        marker="^",
        markersize=7,
        markerfacecolor=(0.5, 0.5, 0.5, 0.35),
        markeredgecolor="0.35",
        markeredgewidth=0.9,
        label="extrapolated (no line)",
    ),
    plt.Line2D([0], [0], color="black", linestyle=":", label="theory (fitted prefactor)"),
]
eps_handles = [
    plt.Line2D([0], [0], color=EPS_COLORS[float(ep)], linewidth=2.0, label=rf"$\epsilon_p={ep:g}$")
    for ep in amplitudes
]

for method, fig, axes in fig_axes_list:
    ax_arr = np.atleast_1d(axes).ravel()
    for ax in ax_arr:
        ax.set_ylim(y_min, y_max)
    ax_arr[0].legend(handles=handles_sem, loc="upper left", frameon=True, fontsize=7.5)
    ax_arr[2].legend(
        handles=eps_handles,
        title=r"Noise levels",
        loc="upper left",
        frameon=True,
        fontsize=7.5,
        ncol=1,
    )
    fig.tight_layout()
    stem = f"fixedacc_scaling_validation_{method}_target0p8"
    out_png = SCALING_DIR / f"{stem}.png"
    out_pdf = SCALING_DIR / f"{stem}.pdf"
    fig.savefig(out_png, dpi=280, bbox_inches="tight")
    fig.savefig(out_pdf, dpi=280, bbox_inches="tight")
    print("saved:", out_png)
    print("saved:", out_pdf)


Bad key axes.grid.alpha in file /Users/krzywdaja/Documents/noisy-learning-advantage/figures_notebooks/single_column.mplstyle, line 30 ('axes.grid.alpha   : 0.35')
You probably need to get an updated matplotlibrc file from
https://github.com/matplotlib/matplotlib/blob/v3.9.4/lib/matplotlib/mpl-data/matplotlibrc
or from the matplotlib source distribution


Hypergraph | relaxation | fitted ĉ per ε_p: $\epsilon_p=0.01\rightarrow\hat{c}=10.57$ ; $\epsilon_p=0.05\rightarrow\hat{c}=3.624$ ; $\epsilon_p=0.1\rightarrow\hat{c}=2.965$
saved: /Users/krzywdaja/Documents/noisy-learning-advantage/curve_fitting/plots_fixed_acc_validation_from_B1600/scaling_validation/fixedacc_scaling_validation_hypergraph_target0p8.png
saved: /Users/krzywdaja/Documents/noisy-learning-advantage/curve_fitting/plots_fixed_acc_validation_from_B1600/scaling_validation/fixedacc_scaling_validation_hypergraph_target0p8.pdf
saved: /Users/krzywdaja/Documents/noisy-learning-advantage/curve_fitting/plots_fixed_acc_validation_from_B1600/scaling_validation/fixedacc_scaling_validation_eigenshadow_target0p8.png
saved: /Users/krzywdaja/Documents/noisy-learning-advantage/curve_fitting/plots_fixed_acc_validation_from_B1600/scaling_validation/fixedacc_scaling_validation_eigenshadow_target0p8.pdf
